# 🧬 NonBScanner - Calculation Methods Explained

**Interactive notebook showing how calculations are performed for each Non-B DNA subclass**

This notebook provides a detailed breakdown of:
1. **How each detector identifies motifs**
2. **Step-by-step scoring calculations**
3. **Component-level analysis** (stems, loops, arms, tracts, etc.)
4. **Scientific references** for each detection method

---

## Quick Start: Enter Your Sequence

Run the cell below to analyze your sequence and see detailed calculation breakdowns.

In [ ]:
# ============================================================================
# USER INPUT: Enter your DNA sequence here
# ============================================================================

# Option 1: Enter sequence directly
input_sequence = """
GGGTTAGGGTTAGGGTTAGGGAAAAATTTTCGCGCGCGCGATATATATATCCCCTAACCCTAACCCTAACCC
"""

# Option 2: Load from FASTA file (uncomment and modify path)
# from utilities import read_fasta_file
# sequences = read_fasta_file("your_file.fasta")
# input_sequence = list(sequences.values())[0]

# Clean up sequence
sequence = input_sequence.strip().upper().replace('\n', '').replace(' ', '')
sequence_name = "user_input"

print(f"📋 Sequence loaded: {len(sequence)} bp")
print(f"   Preview: {sequence[:50]}..." if len(sequence) > 50 else f"   Sequence: {sequence}")

---

## Step 1: Initialize Detectors and Run Analysis

First, we'll import all the detector classes and run the analysis to identify motifs.

In [ ]:
# Import all detectors
from detectors import (
    CurvedDNADetector,
    SlippedDNADetector,
    CruciformDetector,
    RLoopDetector,
    TriplexDetector,
    GQuadruplexDetector,
    IMotifDetector,
    ZDNADetector,
    APhilicDetector
)
from scanner import analyze_sequence
import pandas as pd
import numpy as np
import re
from collections import Counter

# Initialize all detectors
detectors = {
    'Curved DNA': CurvedDNADetector(),
    'Slipped DNA': SlippedDNADetector(),
    'Cruciform': CruciformDetector(),
    'R-Loop': RLoopDetector(),
    'Triplex': TriplexDetector(),
    'G-Quadruplex': GQuadruplexDetector(),
    'i-Motif': IMotifDetector(),
    'Z-DNA': ZDNADetector(),
    'A-Philic': APhilicDetector()
}

# Run full analysis
all_motifs = analyze_sequence(sequence, sequence_name)

print(f"\n✅ Analysis Complete!")
print(f"   Total motifs detected: {len(all_motifs)}")

# Show summary by class
class_counts = Counter([m.get('Class', 'Unknown') for m in all_motifs])
print("\n📊 Motifs by Class:")
for cls, count in sorted(class_counts.items()):
    print(f"   {cls}: {count}")

---

## Step 2: Detailed Calculation Breakdowns by Class

Below we explain how each detector calculates scores and identifies motifs.

### 🔷 Class 1: Curved DNA Detection

**Detection Method:** A-tract phasing analysis (Koo 1986, Olson 1998)

**Subclasses:**
- **Global Curvature:** A-phased repeats (APRs) with 3+ A-tracts spaced ~11bp apart
- **Local Curvature:** Long A-tracts (≥7bp) or T-tracts (≥7bp)

**Calculation Steps:**
1. Find A/T-rich windows in the sequence
2. Within each window, calculate maxATlen (longest A or AnTn run) and maxTlen (longest T-only run)
3. If (maxATlen - maxTlen) ≥ 3, it's a bona fide A-tract
4. Find the center position of each A-tract
5. Group A-tract centers that are 9.9-11.1bp apart (phased)
6. Calculate phasing score based on deviation from ideal 11bp spacing

In [ ]:
# Curved DNA Calculation Breakdown
print("═" * 80)
print("🔷 CURVED DNA CALCULATION BREAKDOWN")
print("═" * 80)

curved_detector = detectors['Curved DNA']

# Get detailed annotation
curved_annotation = curved_detector.annotate_sequence(sequence)

print("\n📌 Step 1: Find A/T Windows")
print(f"   Found {len(curved_annotation.get('a_tract_windows', []))} A/T-rich windows")

# Show first few windows
for i, window in enumerate(curved_annotation.get('a_tract_windows', [])[:3]):
    print(f"\n   Window {i+1}: Position {window['start']}-{window['end']}")
    print(f"   └─ Sequence: {window['window_seq'][:30]}..." if len(window['window_seq']) > 30 else f"   └─ Sequence: {window['window_seq']}")
    print(f"   └─ maxATlen: {window['maxATlen']} (longest A or AnTn run)")
    print(f"   └─ maxTlen: {window['maxTlen']} (longest T-only run)")
    print(f"   └─ Difference (maxATlen - maxTlen): {window['diff_forward']}")
    print(f"   └─ Is A-tract: {'YES ✓' if window['call'] else 'NO ✗'} (threshold: ≥3)")
    if window['call'] and window['a_center']:
        print(f"   └─ A-tract center position: {window['a_center']:.1f}")

print("\n📌 Step 2: Identify A-Phased Repeats (APRs)")
aprs = curved_annotation.get('aprs', [])
print(f"   Found {len(aprs)} APRs (groups of 3+ phased A-tracts)")

for i, apr in enumerate(aprs[:2]):
    print(f"\n   APR {i+1}:")
    print(f"   └─ A-tract centers: {[round(c, 1) for c in apr['center_positions']]}")
    print(f"   └─ Spacings between centers: {[round(s, 1) for s in apr.get('spacings', [])]}")
    print(f"   └─ Ideal spacing: 11.0 bp (one helical turn)")
    print(f"   └─ Mean deviation from ideal: {apr.get('mean_deviation', 0):.2f} bp")
    print(f"   └─ Phasing score: {apr['score']:.4f}")
    print(f"   └─ Score calculation: 1.0 - (mean_deviation / max_allowed_deviation)")

print("\n📌 Step 3: Find Long A/T Tracts (Local Curvature)")
long_tracts = curved_annotation.get('long_tracts', [])
print(f"   Found {len(long_tracts)} long tracts (≥7bp)")

for i, tract in enumerate(long_tracts[:3]):
    print(f"\n   Tract {i+1}: {tract['base']}-tract at {tract['start']}-{tract['end']}")
    print(f"   └─ Sequence: {tract['seq']}")
    print(f"   └─ Length: {tract['len']} bp")
    print(f"   └─ Score: {tract['score']:.4f}")
    print(f"   └─ Score formula: length / (length + 6)")

# Show detected motifs
curved_motifs = curved_detector.detect_motifs(sequence, sequence_name)
print(f"\n📌 Final Results: {len(curved_motifs)} Curved DNA motifs detected")
for motif in curved_motifs:
    print(f"   • {motif['Subclass']} at {motif['Start']}-{motif['End']}, Score: {motif['Score']}")

### 🔷 Class 2: Slipped DNA Detection

**Detection Method:** K-mer seed-and-extend approach

**Subclasses:**
- **Direct Repeats:** Unit length 10-300bp, spacer ≤10bp
- **STR (Short Tandem Repeats):** Unit size 1-9bp, tandem copies

**Calculation Steps:**
1. Build k-mer index (k=10 for direct repeats)
2. For each k-mer with multiple occurrences, check for extended matches
3. Calculate unit length, spacer length, and GC content
4. For STRs, count consecutive repeat copies

In [ ]:
# Slipped DNA Calculation Breakdown
print("═" * 80)
print("🔷 SLIPPED DNA CALCULATION BREAKDOWN")
print("═" * 80)

slipped_detector = detectors['Slipped DNA']
slipped_motifs = slipped_detector.detect_motifs(sequence, sequence_name)

# Separate by subclass
direct_repeats = [m for m in slipped_motifs if 'Direct' in m.get('Subclass', '')]
strs = [m for m in slipped_motifs if 'STR' in m.get('Subclass', '') or 'unit_' in m.get('Subclass', '')]

print("\n📌 Direct Repeat Detection:")
print(f"   Found {len(direct_repeats)} direct repeats")
for i, dr in enumerate(direct_repeats[:3]):
    print(f"\n   Direct Repeat {i+1}:")
    print(f"   └─ Position: {dr['Start']}-{dr['End']}")
    print(f"   └─ Length: {dr['Length']} bp")
    if 'Unit_Length' in dr:
        print(f"   └─ Unit Length: {dr['Unit_Length']} bp")
    if 'Spacer' in dr:
        print(f"   └─ Spacer Length: {dr['Spacer']} bp")
    if 'Left_Unit' in dr:
        print(f"   └─ Left Unit: {dr['Left_Unit'][:20]}..." if len(dr.get('Left_Unit', '')) > 20 else f"   └─ Left Unit: {dr.get('Left_Unit', '')}")
        print(f"   └─ Right Unit: {dr['Right_Unit'][:20]}..." if len(dr.get('Right_Unit', '')) > 20 else f"   └─ Right Unit: {dr.get('Right_Unit', '')}")
    print(f"   └─ GC Content: {dr.get('GC_Total', 'N/A')}%")

print("\n📌 Short Tandem Repeat (STR) Detection:")
print(f"   Found {len(strs)} STRs")
for i, str_motif in enumerate(strs[:3]):
    print(f"\n   STR {i+1}:")
    print(f"   └─ Position: {str_motif['Start']}-{str_motif['End']}")
    if 'Repeat_Unit' in str_motif:
        print(f"   └─ Repeat Unit: {str_motif['Repeat_Unit']}")
    if 'Unit_Length' in str_motif:
        print(f"   └─ Unit Length: {str_motif['Unit_Length']} bp")
    if 'Number_of_Copies' in str_motif or 'Copies' in str_motif:
        copies = str_motif.get('Number_of_Copies', str_motif.get('Copies', 0))
        print(f"   └─ Number of Copies: {copies}")
    print(f"   └─ Total Length: {str_motif['Length']} bp")
    print(f"   └─ Score: {str_motif['Score']}")
    print(f"   └─ Score formula: Based on repeat perfection and length")

### 🔷 Class 3: Cruciform (Inverted Repeat) Detection

**Detection Method:** K-mer index with reverse complement matching

**Subclasses:**
- **Inverted Repeats:** Arms ≥6bp, loop ≤100bp

**Calculation Steps:**
1. Build k-mer index and precompute reverse complements
2. Find positions where a k-mer and its reverse complement occur
3. Extend matches to find full palindromic arms
4. Calculate arm length, loop length, and GC content

In [ ]:
# Cruciform Calculation Breakdown
print("═" * 80)
print("🔷 CRUCIFORM (INVERTED REPEAT) CALCULATION BREAKDOWN")
print("═" * 80)

cruciform_detector = detectors['Cruciform']
cruciform_motifs = cruciform_detector.detect_motifs(sequence, sequence_name)

print(f"\n📌 Inverted Repeat Detection:")
print(f"   Found {len(cruciform_motifs)} cruciform-forming sequences")

for i, motif in enumerate(cruciform_motifs[:3]):
    print(f"\n   Cruciform {i+1}:")
    print(f"   └─ Position: {motif['Start']}-{motif['End']}")
    print(f"   └─ Total Length: {motif['Length']} bp")
    if 'Arm_Length' in motif:
        print(f"   └─ Arm Length: {motif['Arm_Length']} bp")
    if 'Loop_Length' in motif or 'Loop' in motif:
        loop_len = motif.get('Loop_Length', motif.get('Loop', 0))
        print(f"   └─ Loop Length: {loop_len} bp")
    if 'Left_Arm' in motif:
        print(f"   └─ Left Arm: {motif['Left_Arm'][:20]}..." if len(motif.get('Left_Arm', '')) > 20 else f"   └─ Left Arm: {motif.get('Left_Arm', '')}")
    if 'Right_Arm' in motif:
        print(f"   └─ Right Arm (RC): {motif['Right_Arm'][:20]}..." if len(motif.get('Right_Arm', '')) > 20 else f"   └─ Right Arm (RC): {motif.get('Right_Arm', '')}")
    if 'Loop_Seq' in motif:
        loop_seq = motif['Loop_Seq']
        print(f"   └─ Loop Sequence: {loop_seq[:20]}..." if len(loop_seq) > 20 else f"   └─ Loop Sequence: {loop_seq}")
    print(f"   └─ Score: {motif['Score']}")
    print(f"   └─ Cruciform stability: Based on arm length, GC content, loop size")

### 🔷 Class 4: R-Loop Detection

**Detection Method:** QmRLFS algorithm (Jenjaroenpun & Wongsurawat, 2016)

**Subclasses:**
- **QmRLFS-m1:** RIZ with 3+ G-tracts
- **QmRLFS-m2:** RIZ with 4+ G-tracts (stricter)

**Calculation Steps:**
1. Find R-loop Initiating Zones (RIZ): G-rich regions with G% ≥40%
2. Count G-tracts (runs of 3+ or 4+ Gs) within RIZ
3. Look for R-loop Extending Zones (REZ) downstream
4. Calculate combined score from RIZ G% and REZ properties

In [ ]:
# R-Loop Calculation Breakdown
print("═" * 80)
print("🔷 R-LOOP CALCULATION BREAKDOWN")
print("═" * 80)

rloop_detector = detectors['R-Loop']
rloop_motifs = rloop_detector.detect_motifs(sequence, sequence_name)

# Also get raw annotation for more details
rloop_annotation = rloop_detector.annotate_sequence(sequence)

print(f"\n📌 QmRLFS Detection Results:")
print(f"   Found {len(rloop_annotation)} potential R-loop forming sequences")

for i, ann in enumerate(rloop_annotation[:3]):
    print(f"\n   R-Loop {i+1}:")
    print(f"   └─ Model: {ann.get('model', 'Unknown')}")
    print(f"   └─ Total Position: {ann.get('total_start', 0)+1}-{ann.get('total_end', 0)}")
    print(f"   └─ Total Length: {ann.get('total_length', 0)} bp")
    
    # RIZ (R-loop Initiating Zone) details
    print(f"   \n   RIZ (R-loop Initiating Zone):")
    print(f"   └─ Position: {ann.get('riz_start', 0)+1}-{ann.get('riz_end', 0)}")
    print(f"   └─ Length: {ann.get('riz_length', 0)} bp")
    print(f"   └─ G Percentage: {ann.get('riz_perc_g', 0):.1f}% (threshold: ≥40%)")
    print(f"   └─ Total Gs: {ann.get('riz_g_total', 0)}")
    print(f"   └─ 3G Tracts (GGG+): {ann.get('riz_3gs_count', 0)}")
    print(f"   └─ 4G Tracts (GGGG+): {ann.get('riz_4gs_count', 0)}")
    
    # REZ (R-loop Extending Zone) details
    if ann.get('rez_length', 0) > 0:
        print(f"   \n   REZ (R-loop Extending Zone):")
        print(f"   └─ Position: {ann.get('rez_start', 0)+1}-{ann.get('rez_end', 0)}")
        print(f"   └─ Length: {ann.get('rez_length', 0)} bp")
        print(f"   └─ G Percentage: {ann.get('rez_perc_g', 0):.1f}%")
        print(f"   └─ Linker Length: {ann.get('linker_length', 0)} bp")

print(f"\n📌 Final Motifs: {len(rloop_motifs)} R-Loop motifs detected")

### 🔷 Class 5: Triplex Detection

**Detection Method:** Mirror repeat detection with purine/pyrimidine filtering (Frank-Kamenetskii 1995)

**Subclasses:**
- **Mirror Repeat:** Arms ≥10bp, loop ≤100bp, >90% purine OR pyrimidine
- **Sticky DNA:** (GAA)n or (TTC)n repeats ≥12bp

**Calculation Steps:**
1. Find mirror repeats (left arm = reverse of right arm)
2. Calculate purine (AG) and pyrimidine (CT) content in arms
3. Only accept if >90% homopurine or homopyrimidine
4. Detect GAA/TTC repeats for sticky DNA

In [ ]:
# Triplex Calculation Breakdown
print("═" * 80)
print("🔷 TRIPLEX CALCULATION BREAKDOWN")
print("═" * 80)

triplex_detector = detectors['Triplex']
triplex_motifs = triplex_detector.detect_motifs(sequence, sequence_name)

# Get detailed annotation
triplex_annotation = triplex_detector.annotate_sequence(sequence)

mirror_repeats = [m for m in triplex_motifs if 'mirror' in m.get('Subclass', '').lower()]
sticky_dna = [m for m in triplex_motifs if 'Sticky' in m.get('Subclass', '') or 'GAA' in m.get('Subclass', '') or 'TTC' in m.get('Subclass', '')]

print(f"\n📌 Mirror Repeat Detection:")
print(f"   Found {len(mirror_repeats)} triplex-forming mirror repeats")

for i, motif in enumerate(mirror_repeats[:2]):
    print(f"\n   Mirror Repeat {i+1}:")
    print(f"   └─ Position: {motif['Start']}-{motif['End']}")
    print(f"   └─ Total Length: {motif['Length']} bp")
    if 'Arm_Length' in motif:
        print(f"   └─ Arm Length: {motif['Arm_Length']} bp")
    if 'Loop_Length' in motif:
        print(f"   └─ Loop Length: {motif['Loop_Length']} bp")
    if 'Left_Arm' in motif:
        left_arm = motif['Left_Arm']
        pur_count = sum(1 for b in left_arm if b in 'AG')
        pyr_count = sum(1 for b in left_arm if b in 'CT')
        print(f"   └─ Left Arm: {left_arm[:20]}..." if len(left_arm) > 20 else f"   └─ Left Arm: {left_arm}")
        print(f"   └─ Purine content (AG): {pur_count}/{len(left_arm)} = {pur_count/len(left_arm)*100:.1f}%")
        print(f"   └─ Pyrimidine content (CT): {pyr_count}/{len(left_arm)} = {pyr_count/len(left_arm)*100:.1f}%")
    print(f"   └─ Score: {motif['Score']}")
    print(f"   └─ Triplex requirement: >90% purine OR >90% pyrimidine")

print(f"\n📌 Sticky DNA Detection:")
print(f"   Found {len(sticky_dna)} sticky DNA motifs (GAA/TTC repeats)")

for i, motif in enumerate(sticky_dna[:2]):
    print(f"\n   Sticky DNA {i+1}:")
    print(f"   └─ Position: {motif['Start']}-{motif['End']}")
    print(f"   └─ Sequence: {motif.get('Sequence', '')[:30]}..." if len(motif.get('Sequence', '')) > 30 else f"   └─ Sequence: {motif.get('Sequence', '')}")
    gaa_count = motif.get('Sequence', '').count('GAA')
    ttc_count = motif.get('Sequence', '').count('TTC')
    print(f"   └─ GAA repeats: {gaa_count}")
    print(f"   └─ TTC repeats: {ttc_count}")
    print(f"   └─ Score: {motif['Score']}")

### 🔷 Class 6: G-Quadruplex Detection

**Detection Method:** G4Hunter algorithm (Bedrat et al., 2016) with pattern matching

**Subclasses:**
- **Canonical G4:** 4 G-tracts (≥3 Gs each) with loops 1-7bp
- **Relaxed G4:** 4 G-tracts (≥2 Gs each) with loops 1-12bp
- **Long-loop G4:** First loop 8-15bp
- **Bulged G4:** Loops 8-25bp
- **Multimeric G4:** 5+ G-tracts
- **Imperfect G4:** Interrupted G-tracts
- **G-Triplex:** 3 G-tracts (intermediate)

**Calculation Steps:**
1. Match pattern to identify G-tract structures
2. Calculate G4Hunter score using sliding window
3. Add tract bonus for multiple G-tracts
4. Apply GC balance penalty if C-rich

In [ ]:
# G-Quadruplex Calculation Breakdown
print("═" * 80)
print("🔷 G-QUADRUPLEX CALCULATION BREAKDOWN")
print("═" * 80)

g4_detector = detectors['G-Quadruplex']
g4_motifs = g4_detector.detect_motifs(sequence, sequence_name)

# Get annotation for detailed scoring
g4_annotation = g4_detector.annotate_sequence(sequence)

print(f"\n📌 G-Quadruplex Detection:")
print(f"   Found {len(g4_motifs)} G-quadruplex motifs")

# Group by subclass
g4_by_subclass = {}
for motif in g4_motifs:
    subclass = motif.get('Subclass', 'Unknown')
    if subclass not in g4_by_subclass:
        g4_by_subclass[subclass] = []
    g4_by_subclass[subclass].append(motif)

for subclass, motifs in g4_by_subclass.items():
    print(f"\n   {subclass}: {len(motifs)} motif(s)")
    
    for i, motif in enumerate(motifs[:2]):
        print(f"\n   G4 {i+1} ({subclass}):")
        print(f"   └─ Position: {motif['Start']}-{motif['End']}")
        print(f"   └─ Length: {motif['Length']} bp")
        print(f"   └─ Sequence: {motif.get('Sequence', '')[:40]}..." if len(motif.get('Sequence', '')) > 40 else f"   └─ Sequence: {motif.get('Sequence', '')}")
        
        # Show component details
        if 'Stems' in motif and motif['Stems']:
            print(f"   └─ G-Stems: {motif['Stems']}")
            print(f"   └─ Number of Stems: {len(motif['Stems'])}")
            print(f"   └─ Stem Lengths: {[len(s) for s in motif['Stems']]}")
        if 'Loops' in motif and motif['Loops']:
            print(f"   └─ Loops: {motif['Loops']}")
            print(f"   └─ Loop Lengths: {[len(l) for l in motif['Loops']]}")
        
        print(f"   └─ Score: {motif['Score']}")
        print(f"   └─ GC Content: {motif.get('GC_Total', 'N/A')}%")

# Explain G4Hunter scoring
print("\n📌 G4Hunter Score Calculation:")
print("   1. Sliding window (default 25bp)")
print("   2. For each base: G=+1, C=-1, A/T=0")
print("   3. Find max absolute window sum")
print("   4. Normalized window score = max_abs / window_size")
print("   5. Tract bonus: +0.08 per additional G-tract beyond 2")
print("   6. GC penalty: -0.1 to -0.2 if C-rich (GC balance < -0.1)")
print("   7. Final score = (normalized + tract_bonus - penalty) × (length/window)")

### 🔷 Class 7: i-Motif Detection

**Detection Method:** C-tract pattern matching with Hur AC-motif detection

**Subclasses:**
- **Canonical i-Motif:** 4 C-tracts (≥3 Cs each) with loops 1-7bp
- **Relaxed i-Motif:** 4 C-tracts (≥2 Cs each) with loops 1-12bp
- **HUR AC-Motif:** A-C alternating patterns (A3-loop-C3-loop-C3-loop-C3)

**Calculation Steps:**
1. Find C-tract patterns matching i-motif structure
2. Calculate C-tract density and count
3. For HUR AC-motifs, check A-C alternating pattern
4. Score based on tract lengths and density

In [ ]:
# i-Motif Calculation Breakdown
print("═" * 80)
print("🔷 i-MOTIF CALCULATION BREAKDOWN")
print("═" * 80)

imotif_detector = detectors['i-Motif']
imotif_motifs = imotif_detector.detect_motifs(sequence, sequence_name)

# Get detailed annotation
imotif_annotation = imotif_detector.annotate_sequence(sequence)

print(f"\n📌 i-Motif Detection:")
print(f"   Found {len(imotif_motifs)} i-Motif structures")

for i, motif in enumerate(imotif_motifs[:3]):
    print(f"\n   i-Motif {i+1} ({motif.get('Subclass', 'Unknown')}):")
    print(f"   └─ Position: {motif['Start']}-{motif['End']}")
    print(f"   └─ Length: {motif['Length']} bp")
    print(f"   └─ Sequence: {motif.get('Sequence', '')[:40]}..." if len(motif.get('Sequence', '')) > 40 else f"   └─ Sequence: {motif.get('Sequence', '')}")
    
    # Show C-tract components
    if 'Stems' in motif and motif['Stems']:
        print(f"   └─ C-Stems: {motif['Stems']}")
        print(f"   └─ Number of C-Stems: {len(motif['Stems'])}")
        print(f"   └─ Stem Lengths: {[len(s) for s in motif['Stems']]}")
    if 'Loops' in motif and motif['Loops']:
        print(f"   └─ Loops: {motif['Loops']}")
    
    # Calculate C density
    seq = motif.get('Sequence', '')
    if seq:
        c_count = seq.count('C')
        c_density = c_count / len(seq) * 100
        print(f"   └─ C Content: {c_count}/{len(seq)} = {c_density:.1f}%")
    
    print(f"   └─ Score: {motif['Score']}")

# Explain scoring
print("\n📌 i-Motif Score Calculation:")
print("   1. Count C-tracts (CC or longer)")
print("   2. Calculate C density = total_C_length / sequence_length")
print("   3. Tract bonus = 0.12 × (num_c_tracts - 2)")
print("   4. Final score = C_density + tract_bonus (capped at 1.0)")

### 🔷 Class 8: Z-DNA Detection

**Detection Method:** 10-mer scoring table (Ho et al., 1986)

**Subclasses:**
- **Z-DNA:** Alternating purine-pyrimidine (CG/GC rich)
- **eGZ (Extruded-G) DNA:** CG-rich regions

**Calculation Steps:**
1. Scan for all 10-mer matches in the sequence
2. Look up score for each 10-mer in the scoring table
3. Merge overlapping/adjacent 10-mer matches into regions
4. Distribute each 10-mer's score across its 10 bases
5. Sum per-base scores for final region score

In [ ]:
# Z-DNA Calculation Breakdown
print("═" * 80)
print("🔷 Z-DNA CALCULATION BREAKDOWN")
print("═" * 80)

zdna_detector = detectors['Z-DNA']
zdna_motifs = zdna_detector.detect_motifs(sequence, sequence_name)

# Get annotation for detailed breakdown
zdna_annotation = zdna_detector.annotate_sequence(sequence)

print(f"\n📌 Z-DNA Detection:")
print(f"   Found {len(zdna_motifs)} Z-DNA regions")

# Show 10-mer scoring table examples
print("\n📌 10-mer Scoring Table (examples):")
example_10mers = ['GCGCGCGCGC', 'CGCGCGCGCG', 'ACGCGCGCGC', 'TGCGCGCGCG']
for tenmer in example_10mers:
    score = zdna_detector.TENMER_SCORE.get(tenmer, 0)
    print(f"   {tenmer}: {score}")

for i, ann in enumerate(zdna_annotation[:3]):
    print(f"\n   Z-DNA Region {i+1}:")
    print(f"   └─ Position: {ann['start']+1}-{ann['end']}")
    print(f"   └─ Length: {ann['length']} bp")
    print(f"   └─ Contributing 10-mers: {ann['n_10mers']}")
    print(f"   └─ Sum Score: {ann['sum_score']:.2f}")
    print(f"   └─ Mean 10-mer Score: {ann['mean_score_per10mer']:.2f}")
    
    # Show contributing 10-mers
    if 'contributing_10mers' in ann and ann['contributing_10mers']:
        print(f"   └─ 10-mers detected:")
        for j, tenmer_info in enumerate(ann['contributing_10mers'][:3]):
            print(f"      {j+1}. {tenmer_info['tenmer']} at pos {tenmer_info['start']+1}, score: {tenmer_info['score']}")

# Explain scoring
print("\n📌 Z-DNA Score Calculation:")
print("   1. Find all 10-mers that match the scoring table")
print("   2. Merge overlapping/adjacent matches into regions")
print("   3. Distribute each 10-mer's score equally across 10 bases")
print("   4. Sum per-base contributions for region score")
print("   5. Threshold: sum_score > 50.0 for reporting")

### 🔷 Class 9: A-Philic DNA Detection

**Detection Method:** Tetranucleotide analysis (Gorin 1995)

**Subclasses:**
- **A-philic DNA:** A-rich regions with specific tetranucleotide patterns

**Calculation Steps:**
1. Scan for A-philic tetranucleotides (e.g., AAAA, AAAT, AATA...)
2. Merge overlapping matches into regions
3. Calculate A-content and tract properties
4. Score based on A-tract length and density

In [ ]:
# A-Philic DNA Calculation Breakdown
print("═" * 80)
print("🔷 A-PHILIC DNA CALCULATION BREAKDOWN")
print("═" * 80)

aphilic_detector = detectors['A-Philic']
aphilic_motifs = aphilic_detector.detect_motifs(sequence, sequence_name)

# Get annotation
aphilic_annotation = aphilic_detector.annotate_sequence(sequence)

print(f"\n📌 A-Philic DNA Detection:")
print(f"   Found {len(aphilic_motifs)} A-philic regions")

for i, motif in enumerate(aphilic_motifs[:3]):
    print(f"\n   A-Philic Region {i+1}:")
    print(f"   └─ Position: {motif['Start']}-{motif['End']}")
    print(f"   └─ Length: {motif['Length']} bp")
    print(f"   └─ Sequence: {motif.get('Sequence', '')[:40]}..." if len(motif.get('Sequence', '')) > 40 else f"   └─ Sequence: {motif.get('Sequence', '')}")
    
    # Calculate A content
    seq = motif.get('Sequence', '')
    if seq:
        a_count = seq.count('A')
        a_content = a_count / len(seq) * 100
        print(f"   └─ A Content: {a_count}/{len(seq)} = {a_content:.1f}%")
        
        # Find A-tracts
        a_tracts = re.findall(r'A{3,}', seq)
        if a_tracts:
            print(f"   └─ A-tracts found: {len(a_tracts)}")
            print(f"   └─ A-tract lengths: {[len(t) for t in a_tracts]}")
    
    print(f"   └─ Score: {motif['Score']}")

print("\n📌 A-Philic Score Calculation:")
print("   1. Scan for A-philic tetranucleotides")
print("   2. Merge overlapping matches")
print("   3. Calculate A content and tract density")
print("   4. Score = A_content × 0.7 + tract_bonus × 0.3")

---

## Summary: All Detected Motifs with Calculations

The table below shows all detected motifs with their key calculation metrics.

In [ ]:
# Summary Table
print("═" * 100)
print("📊 COMPLETE MOTIF SUMMARY WITH CALCULATIONS")
print("═" * 100)

if all_motifs:
    # Create DataFrame for display
    summary_data = []
    for motif in all_motifs:
        row = {
            'Class': motif.get('Class', ''),
            'Subclass': motif.get('Subclass', ''),
            'Start': motif.get('Start', 0),
            'End': motif.get('End', 0),
            'Length': motif.get('Length', 0),
            'Score': motif.get('Score', 0),
            'Method': motif.get('Method', '')
        }
        summary_data.append(row)
    
    df_summary = pd.DataFrame(summary_data)
    display(df_summary)
    
    # Statistics
    print("\n📈 Statistics:")
    print(f"   Total Motifs: {len(all_motifs)}")
    print(f"   Average Score: {df_summary['Score'].mean():.3f}")
    print(f"   Average Length: {df_summary['Length'].mean():.1f} bp")
    print(f"   Classes Detected: {df_summary['Class'].nunique()}")
    print(f"   Subclasses Detected: {df_summary['Subclass'].nunique()}")
else:
    print("No motifs detected in the input sequence.")
    print("\nTry a sequence with known Non-B DNA structures, for example:")
    print("   G-Quadruplex: GGGTTAGGGTTAGGGTTAGGG")
    print("   i-Motif: CCCCTAACCCTAACCCTAACCC")
    print("   Z-DNA: CGCGCGCGCGCGCG")

---

## 📚 Scientific References

| Motif Class | Method | Key References |
|-------------|--------|----------------|
| Curved DNA | A-tract phasing | Koo 1986, Olson 1998, Crothers 1992 |
| Slipped DNA | K-mer indexing | Schlötterer 2000, Weber 1989 |
| Cruciform | Inverted repeat detection | Lilley 2000, Pearson 1996 |
| R-Loop | QmRLFS algorithm | Jenjaroenpun 2016, Aguilera 2012 |
| Triplex | Mirror repeat + purine filter | Frank-Kamenetskii 1995, Sakamoto 1999 |
| G-Quadruplex | G4Hunter | Bedrat 2016, Burge 2006, Todd 2005 |
| i-Motif | C-tract analysis | Gehring 1993, Zeraati 2018 |
| Z-DNA | 10-mer scoring | Ho 1986, Rich 1984 |
| A-Philic | Tetranucleotide analysis | Gorin 1995, Nelson 1987 |